In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime as dt
from datetime import timedelta

base = pd.read_csv('F26/f26_wbl.csv', parse_dates=['timestamp'])

# drop the rows before 10, 25, 22:52:44 and after 10, 26, 00:19:11, when the balloon burst
base = base[(base.timestamp.dt.tz_localize(None) >= dt(2026, 11, 14, 22, 41, 29))]
# drop unused columns


### PREPARE DATA FOR RAOB FORMAT

def calculate_wind(df):
    R = 6371000  # Earth radius in meters
    
    lat = np.radians(df['lat'].to_numpy())
    lon = np.radians(df['lon'].to_numpy())
    
    # Prepend ONLY the first element (lat[0], lon[0]) to keep shape (N,)
    dlat = np.diff(lat, prepend=lat[0])
    dlon = np.diff(lon, prepend=lon[0])
    
    # Midpoint latitude for accurate East-West scaling
    lat_prev = np.roll(lat, 1)
    lat_prev[0] = lat[0]
    lat_avg = 0.5 * (lat + lat_prev)
    
    # Displacements in meters
    dy = R * dlat                      # Northward displacement (v)
    dx = R * dlon * np.cos(lat_avg)    # Eastward displacement (u)
    
    # Time step in seconds
    dt_sec = df['timestamp'].diff().dt.total_seconds().to_numpy()
    
    # Wind vector components (m/s)
    u = np.divide(dx, dt_sec, out=np.full_like(dx, np.nan), where=(dt_sec > 0))
    v = np.divide(dy, dt_sec, out=np.full_like(dy, np.nan), where=(dt_sec > 0))
    
    # Wind speed in knots (standard RAOB unit) & direction (FROM)
    wind_speed_kts = np.hypot(u, v) * 1.94384
    wind_dir_deg = np.degrees(np.arctan2(-u, -v))
    
    return wind_speed_kts, wind_dir_deg

base['WSPD'], base['WDIR'] = calculate_wind(base)

# calculate dewpoint (in celsius) from temperature and humidity
#  TD: =243.04*(LN(RH/100)+((17.625*T)/(243.04+T)))/(17.625-LN(RH/100)-((17.625*T)/(243.04+T)))
base['DWPT'] = 243.04 * (np.log(base['humidity']/100) + ((17.625 * base['temp']) / (243.04 + base['temp']))) / (17.625 - np.log(base['humidity']/100) - ((17.625 * base['temp']) / (243.04 + base['temp'])))
base['DWPT'] = base['DWPT'].round(2)

# # calculate wind direction from heading
# base['WDIR'] = (base['heading'] + 180) % 360
# base['WDIR'] = base['WDIR'].round(2)

# # convert wind speed from m/s to knots
# base['WSPD'] = base['vel_h'] * 1.94384
# base['WSPD'] = base['WSPD'].round(2)


# change pressure to LEVEL
base['LEVEL'] = base['pressure']
base['LEVEL'] = base['LEVEL'].round(2)

# change alt to HGHT
base['HGHT'] = base['alt']
base['HGHT'] = base['HGHT'].round(2)
# change temp to TEMP
base['TEMP'] = base['temp']
base['TEMP'] = base['TEMP'].round(2)

final = base[['LEVEL','HGHT','TEMP','DWPT','WDIR','WSPD']]
final

,LEVEL,HGHT,TEMP,DWPT,WDIR,WSPD
1467,1008.6,43,13.1,7.37,NaN,NaN
1468,1008.3,48,13.4,7.09,16.156142,6.750971
1469,1007.9,52,13.8,6.90,-49.206776,9.925067
1470,1007.4,59,13.9,6.76,-49.206785,9.925068
1471,1006.9,66,14.0,6.67,-40.993631,5.727349
...,...,...,...,...,...,...
2408,404.9,7211,-23.5,-24.98,-115.457280,60.342308
2409,401.6,7271,-24.1,-25.48,-113.310100,52.983794
2410,398.6,7327,-24.6,-25.74,-112.850671,50.094064
2411,397.7,7340,-24.7,-25.77,-98.059497,61.667074


In [2]:
# save to csv
final.to_csv('20261114_iMet1_WBL_filtered_raob.txt', index=False)